#**[ LangChain, RAG 되돌아보기 ]**

#1 . LangChain

In [ ]:
!pip install -q langchain
!pip install -q langchain-openai

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path = "/content/.env", override=True)

print(".env 내 OPENAI_API_KEY가 환경변수에 할당됐습니다:", os.environ["OPENAI_API_KEY"][:5]+"*****")

## 1-1. LCEL(LangChain Expression Language)

기본 예시: 프롬프트 + 모델 + 출력 파서

가장 기본적이고 일반적인 사용 사례는 prompt 템플릿과 모델을 함께 연결하는 것

```
chain = prompt | model | output_parser
```

`|` 기호는 [unix 파이프 연산자](<https://en.wikipedia.org/wiki/Pipeline_(Unix)>)와 유사하며, 서로 다른 구성 요소를 연결하고 한 구성 요소의 출력을 다음 구성 요소의 입력으로 전달

이 체인에서 사용자 입력은 프롬프트 템플릿으로 전달되고, 그런 다음 프롬프트 템플릿 출력은 모델로 전달

###① 영어회화 (기본 예시)

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model

template = """
당신은 영어를 가르치는 10년차 영어 선생님입니다. 주어진 상황에 맞는 영어 회화를 작성해 주세요.
양식은 [FORMAT]을 참고하여 작성해 주세요.

#상황:
{question}

#FORMAT:
- 영어 회화:
- 한글 해석:
"""

# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다.
model = init_chat_model("gpt-4.1-nano")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

In [ ]:
# 체인을 구성합니다.
chain = prompt | model | output_parser

In [ ]:
# 완성된 Chain을 실행하여 답변을 얻습니다.
print(chain.invoke({"question": "저는 식당에 가서 음식을 주문하고 싶어요"}))

In [ ]:
answer = chain.stream({"question": "저는 식당에 가서 음식을 주문하고 싶어요"})
# 스트리밍 출력
for chunk in answer:
    print(chunk, end="", flush=True)

In [ ]:
answer = chain.stream({"question": "미국에서 피자 주문"})
# 스트리밍 출력
for chunk in answer:
    print(chunk, end="", flush=True)

###② 수도 & 국토면적 (parallel 예시)

In [ ]:
from langchain_core.runnables import RunnableParallel

# {country} 의 수도를 물어보는 체인을 생성합니다.
chain1 = (
    PromptTemplate.from_template("{country} 의 수도는 어디야?")
    | model
    | StrOutputParser()
)

# {country} 의 면적을 물어보는 체인을 생성합니다.
chain2 = (
    PromptTemplate.from_template("{country} 의 면적은 얼마야?")
    | model
    | StrOutputParser()
)

# 위의 2개 체인을 동시에 생성하는 병렬 실행 체인을 생성합니다.
combined = RunnableParallel(capital=chain1, area=chain2)


In [ ]:
# chain1 (수도) 를 실행합니다.
chain1.invoke({"country": "대한민국"})

In [ ]:
# chain2 (국토 면적) 를 실행합니다.
chain2.invoke({"country": "미국"})

In [ ]:
# 병렬 (수도, 국토 면적) 실행 체인을 실행합니다.
combined.invoke({"country": "대한민국"})

In [ ]:
# 배치 처리를 수행합니다.
chain1.batch([{"country": "대한민국"}, {"country": "미국"}])

In [ ]:
# 배치 처리를 수행합니다.
chain2.batch([{"country": "대한민국"}, {"country": "미국"}])

In [ ]:
# 주어진 데이터를 배치로 처리합니다.
combined.batch([{"country": "대한민국"}, {"country": "미국"}])

###③ 질문 분류 & 분기 (Runnable 예시)

RunnableBranch, RunnablePassthrough

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """주어진 사용자 질문을 `수학`, `과학`, 또는 `기타` 중 하나로 분류하세요. 한 단어 이상으로 응답하지 마세요.

<question>
{question}
</question>

Classification:"""
)

# 체인을 생성합니다.
chain = (
    prompt
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()  # 문자열 출력 파서를 사용합니다.
)


In [ ]:
# 질문을 입력하여 체인을 호출합니다.
# chain.invoke({"question": "2+2 는 무엇인가요?"})

chain.invoke({"question": "작용 반작용의 법칙은 무엇인가요?"})

# chain.invoke({"question": "Google은 어떤 회사인가요?"})


In [ ]:
math_chain = (
    PromptTemplate.from_template(
        """You are an expert in math. \
Always answer questions starting with "파스칼선생님께서 말씀하시기를..". \
Respond to the following question:

Question: {question}
Answer:"""
    )
    # OpenAI의 LLM을 사용합니다.
    | ChatOpenAI(model="gpt-4o-mini")
)

science_chain = (
    PromptTemplate.from_template(
        """You are an expert in science. \
Always answer questions starting with "뉴턴선생님께서 말씀하시기를..". \
Respond to the following question:

Question: {question}
Answer:"""
    )
    # OpenAI의 LLM을 사용합니다.
    | ChatOpenAI(model="gpt-4o-mini")
)

general_chain = (
    PromptTemplate.from_template(
        """Respond to the following question concisely:

Question: {question}
Answer:"""
    )
    # OpenAI의 LLM을 사용합니다.
    | ChatOpenAI(model="gpt-4o-mini")
)


In [ ]:
# RunnableBranch 로 분기
from langchain_core.runnables import RunnableBranch

branch = RunnableBranch(
    # 주제에 "수학"이 포함되어 있는 경우, math_chain을 실행합니다.
    (lambda x: "수학" in x["topic"].lower(), math_chain),
    # 주제에 "과학"이 포함되어 있는 경우, science_chain을 실행합니다.
    (lambda x: "과학" in x["topic"].lower(), science_chain),
    # 그 외의 경우 general_chain을 실행합니다.
    general_chain,
)

from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

full_chain = (
    {"topic": chain, "question": RunnablePassthrough()}
    | branch
    | StrOutputParser()
)

In [ ]:
# 수학과 관련된 질문을 입력하여 체인을 호출합니다.
full_chain.invoke({"question": "미적분의 개념에 대해 말씀해 주세요."})


In [ ]:
# 과학과 관련된 질문을 입력하여 체인을 호출합니다.
full_chain.invoke({"question": "중력은 어떻게 작용하나요?"})


In [ ]:
# 기타 질문을 입력하여 체인을 호출합니다.
full_chain.invoke({"question": "RAG(Retrieval Augmented Generation)은 무엇인가요?"})


##1-2. Basic Agent

###① 시간&날씨&수학계산 Agent

In [ ]:
from langchain_core.tools import tool
import datetime
from datetime import timezone, timedelta

# 도구 정의
@tool
def get_current_time() -> str:
    """현재 시간을 반환합니다."""

    # KST는 UTC+9
    kst = timezone(timedelta(hours=9))
    now_kst = datetime.datetime.now(kst)
    return now_kst.strftime("%Y년 %m월 %d일 %H시 %M분")


In [ ]:
@tool
def calculate(expression: str) -> float:
    """수학 계산을 수행합니다.

    Args:
        expression: 계산할 수식 (예: "2 + 3 * 4")
    """
    try:
        result = eval(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"계산 오류: {e}"


In [ ]:
@tool
def get_weather(city: str) -> str:
    """도시의 날씨를 조회합니다. 도시명은 한글로 해줘요

    Args:
        city: 도시 이름 (예: '서울', '부산', '제주')
    """
    # 실제로는 날씨 API 호출
    # 여기서는 예시 데이터 반환
    weather_data = {
        "서울": "맑음 ☀️ 22°C",
        "부산": "흐림 ☁️ 20°C",
        "제주": "비 🌧️ 18°C"
    }
    return weather_data.get(city, f"{city}의 날씨 정보를 찾을 수 없습니다.")


In [ ]:
# 도구 목록
tools = [get_current_time, calculate, get_weather]

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools = tools,
    system_prompt="너는 도구를 확인하고, 적절한 도구를 호출해서 신중하게 답변해야 해"
)

In [ ]:
agent

In [ ]:
agent.invoke({"messages": [{"role": "user", "content": "지금 몇 시야?"}]})

In [ ]:
agent.invoke({"messages": [{"role": "user", "content": "15 곱하기 8은?"}]})

In [ ]:
agent.invoke({"messages": [{"role": "user", "content": "부산 날씨 알려줘"}]})

In [ ]:
# 대화형 질의
print("질문을 입력하세요 (종료: exit or quit)")
while True:
    question = input("\n질문: ")
    if question.lower() in ['exit', 'quit', '종료', '끝']:
        print("종료합니다.")
        break

    response = agent.invoke({"messages": [{"role": "user", "content": question}]})

    # print(f"답변: {response}")
    print(f"답변: {response["messages"][-1].content}")

###② 대화 기억하기

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model,
    tools,
    checkpointer = InMemorySaver(),
)

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요. 저는 Jumany입니다."}]},
    {"configurable": {"thread_id": "thread_1"}},
)

In [ ]:
print(response["messages"][-1].content)

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요. 제 이름이 뭐죠?"}]},
    {"configurable": {"thread_id": "thread_1"}},
 )

In [ ]:
print(response["messages"][-1].content)

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요. 제 이름이 뭐죠?"}]},
    {"configurable": {"thread_id": "thread_2"}},
)

In [ ]:
print(response["messages"][-1].content)

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "지금까지 무슨 얘기 나눴죠?"}]},
    {"configurable": {"thread_id": "thread_1"}},
)

In [ ]:
response

In [ ]:
for i, msg in enumerate(response["messages"]):
    print(f"--- Message {i+1} : {msg.type} ---")
    print(msg.content)
    print()

###③ 이메일 응대 Agent

In [ ]:
from langchain.tools import tool
from typing import List, Dict

# 이메일 전송 도구
@tool
def send_email_tool(to: str, subject: str, body: str) -> str:
    """
    지정한 이메일 주소로 메일을 보내는 도구입니다.
    """
    return f"✅ 이메일이 성공적으로 전송되었습니다.\n수신자: {to}\n제목: {subject}\n내용: {body[:50]}..."


# 이메일 읽기 도구
@tool
def read_email_tool(limit: int = 3) -> List[Dict[str, str]]:
    """
    고객이 온라인 쇼핑몰에 보낸 컴플레인, 문의, 혹은 확인 관련 이메일 1개를 읽는 도구입니다.
    """
    return f"✅ 이메일이 성공적으로 조회되었습니다."

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# Structured Output 정의
class EmailAnalysis(BaseModel):
    """이메일 내용을 분석한 결과 구조."""
    intent: Literal["complaint", "inquiry", "confirmation", "other"] = Field(
        description="이메일의 주요 의도 (예: complaint=불만, inquiry=문의, confirmation=확인, other=기타)"
    )
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="이메일의 감정 상태")
    summary: str = Field(description="이메일 내용 요약")
    next_action: str = Field(description="에이전트가 수행해야 할 다음 단계 (예: 회신, 확인, 무시 등)")

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")
tools = [send_email_tool, read_email_tool]

agent = create_agent(
    model=model,
    tools=tools,
    # 핵심: 최종 산출물의 규격을 EmailAnalysis 스키마로 강제합니다.
    response_format=ToolStrategy(EmailAnalysis),
    middleware=[
        # 이메일 도구가 실제 백엔드에 없더라도 가상으로 동작하게 해주는 실습용 미들웨어
        LLMToolEmulator(model="gpt-5-nano"),
    ],
)


In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "최근 온 메일 확인하고 고객의 의도와 감정, 요약, 그리고 어떤 행동이 필요한지 분석하세요."}]}
)

In [ ]:
response

In [ ]:
response["messages"]

In [ ]:
response["structured_response"]

In [ ]:
analysis = response["structured_response"]

In [ ]:
print(f"\n*intent = {analysis.intent}")
print(f"\n*sentiment = {analysis.sentiment}")
print(f"\n*summaty = {analysis.summary}")
print(f"\n*next action = {analysis.next_action}")

In [ ]:
# Pydantic 객체를 딕셔너리로 변환
data_for_db = analysis.model_dump()

# 구조화된 스키마 데이터를 기반으로 한 자동화 파이프라인 예시
if data_for_db["intent"] == "complaint" and data_for_db["sentiment"] == "negative":
    # 1. CS팀 슬랙 채널에 긴급 알림 전송 (API 호출)
    # 2. Jira 이슈 트래커에 '긴급(High)' 티켓 자동 생성
    print("🚨 [긴급] 불만 접수! CS팀에 즉시 알림을 전송합니다.")
    print(f"요약: {data_for_db['summary']}")
elif data_for_db["intent"] == "inquiry":
    # FAQ 데이터베이스 검색 후 자동 회신 스크립트 실행
    print("ℹ️ 일반 문의 접수. 자동 회신 프로세스를 시작합니다.")


##1-3. Advanced Agent

###① Human-in-the-loop

In [ ]:
# 이메일 전송 도구
# 이메일 읽기 도구

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()  #단기메모리 사용

agent = create_agent(
    model="gpt-5-nano",
    tools=[send_email_tool, read_email_tool],
    checkpointer=checkpointer, # 체크포인터 연결
    middleware=[
        LLMToolEmulator(model="gpt-5-nano"),
        HumanInTheLoopMiddleware(
            interrupt_on={
                # 이메일 보내기 전에 사람의 판단 개입
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                # 이메일 읽기는 자동 승인 (사람 개입 없음)
                "read_email_tool": False,
            }
        ),
    ],
)

이메일을 읽을때는 사람 개입 없음

In [ ]:
prompt = "무슨 메일 왔는지 확인해줘"

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
    {"configurable": {"thread_id": "HIL-a"}}
)

In [ ]:
response["messages"][-1].pretty_print()

이메일 보내기 전에 사람의 판단 개입


```
  "send_email_tool": {
      "allowed_decisions": ["approve", "edit", "reject"],
  },
```

              

In [ ]:
prompt = "교수님한테 내일 찾아뵙겠다는 메일 작성해서 보내줘."

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
    {"configurable": {"thread_id": "HIL-b"}}
)

In [ ]:
response["messages"][-1].pretty_print()

In [ ]:
prompt = """가장 공손하고 격식있는 버전으로 작성하고, 발송 전에 내게 확인을 받고서 발송해줘.
            - 교수님 이메일 : jumany@email.com
            - 소속 학과 : 컴퓨터 공학과
            - 학번 : 20999901
            - 이름 : 빌게이츠
            - 전화번호 : 010-1234-0000
            - 방문 목적 : 취업 상담
            - 방문 시간대 : 모레 오전 9시~12시 """

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
    {"configurable": {"thread_id": "HIL-b"}}
)

In [ ]:
response["messages"][-1].pretty_print()

In [ ]:
approved_response = agent.invoke(
    {"messages": [{"role": "user", "content": "이메일을 확인했어요. 이 이메일을 보내세요."}]},
    {"configurable": {"thread_id": "HIL-b", "decision": "approve"}}
)

In [ ]:
approved_response["messages"][-1].pretty_print()

###② PII Detection

In [ ]:
@tool
def save_customer_feedback(feedback: str) -> str:
    """고객 피드백을 저장하는 도구"""
    return f"📥 고객 피드백 저장 완료: {feedback}"

In [ ]:
from langchain.agents.middleware import LLMToolEmulator, PIIMiddleware

agent = create_agent(
    model="gpt-5-nano",
    tools=[save_customer_feedback],
    middleware=[
        LLMToolEmulator(model="gpt-5-nano"),
        # 이메일 주소는 전부 마스킹 처리
        PIIMiddleware("email", strategy="redact", apply_to_input=True),

        # 카드번호는 마지막 4자리만 남기고 나머지 마스킹 처리
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # API Key 형태(sk-로 시작하는 32자리)는 감지되면 실행 중단
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block", # 차단 (에러 발생)
            apply_to_input=True,
        ),
    ],
)

In [ ]:
prompt = "안녕하세요. 저는 둘리(이메일 : user123@example.com)입니다. 어제 아이폰 구매했는데 결제가 잘 됐는지 확인 부탁합니다."

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
)

In [ ]:
for i, msg in enumerate(response["messages"]):
    print(f"--- Message {i+1} : {msg.type} ---")
    print(msg.content)
    print()

In [ ]:
prompt = "안녕하세요. 저는 둘리(이메일 : user123@example.com)입니다. 제 카드번호는 1234123443214321 입니다. 결제 문제를 해결해주세요."

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
)

In [ ]:
for i, msg in enumerate(response["messages"]):
    print(f"--- Message {i+1} : {msg.type} ---")
    print(msg.content)
    print()

In [ ]:
prompt = "안녕하세요. API 호출이 오류가 나요. 제 API Key는 sk-12345678901234567890123456789012 예요. 확인해주세요"

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
)

Custom PII Detector

In [ ]:
phone_number_detector_regex = r"\b(010)[-\s]?(\d{3,4})[-\s]?(\d{4})\b"

# 커스텀 PII 미들웨어 생성
phone_masking_middleware = PIIMiddleware(
    # pii_type: "phone_number" 라는 커스텀 이름 지정
    pii_type = "phone_number",

    # detector: 위에서 만든 정규식 전달
    detector = phone_number_detector_regex,

    # strategy: "mask" (마스킹)
    # 마스킹은 기본적으로 마지막 4자리를 제외하고 마스킹
    strategy = "mask",

    # apply_to_input: 사용자 입력에 적용
    apply_to_input = True,
)

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[save_customer_feedback],
    middleware=[phone_masking_middleware]
)

In [ ]:
prompt = "안녕하세요, 제 핸드폰 번호는 010-1234-5678 입니다. 등록해주세요."

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
)

In [ ]:
for i, msg in enumerate(response["messages"]):
    print(f"--- Message {i+1} : {msg.type} ---")
    print(msg.content)
    print()

###③ Before Guardrails

In [ ]:
# 차단 및 제재 키워드 정의
forbidden_topics = {
    # 부정행위 관련
    "cheating": ["답지", "정답 알려줘", "숙제 대신", "써줘", "베끼기"],
    # 학습 방해 요소
    "distraction": ["롤", "게임", "유튜브", "아이돌", "웹툰"],
    # 유해 콘텐츠
    "harmful": ["담배", "술", "폭력", "싸움", "바보"]
}

In [ ]:
from langchain.agents.middleware import before_agent

@before_agent(can_jump_to=["end"])
def education_guardrail(state, runtime):
    """
    학생의 질문 의도를 파악하여 교육적이지 않거나 부정행위가 의심될 경우,
    LLM(AI)에게 질문을 넘기지 않고 교육적인 멘트로 즉시 교정합니다.
    """

    # 1. 메시지 유효성 검사
    if not state["messages"]:
        return None

    last_message = state["messages"][-1]
    if last_message.type != "human":
        return None

    user_text = last_message.content

    # 2. 카테고리별 검사 로직
    # 단순히 막는 것을 넘어, '왜' 안되는지 카테고리별로 다른 피드백 주기

    # Case A: 부정행위 방지 (Cheating Prevention)
    # AI가 숙제를 통째로 해주는 것을 방지
    for keyword in forbidden_topics["cheating"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "🚫 스스로 고민해봐야 실력이 늘어요! 정답을 바로 알려드리는 대신, 힌트를 드릴까요? 어떤 부분이 가장 어려운지 말해주세요."
                }],
                "jump_to": "end"
            }

    # Case B: 학습 집중 유도 (Focus Management)
    # 공부 중에 게임이나 딴짓 이야기를 하면 다시 공부로 유도
    for keyword in forbidden_topics["distraction"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "⏰ 지금은 공부에 집중할 시간이에요! 딴짓은 쉬는 시간에 하고, 지금 풀고 있는 문제에 집중해볼까요?"
                }],
                "jump_to": "end"
            }

    # Case C: 유해 콘텐츠 차단 (Safety)
    # 교육 서비스의 브랜드 안전성(Brand Safety)을 위한 기능
    for keyword in forbidden_topics["harmful"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "⚠️ 부적절한 대화 주제입니다. 바르고 고운 말을 사용해주세요."
                }],
                "jump_to": "end"
            }

    # 3. 통과 (Pass)
    # 위 조건들에 걸리지 않으면 정상적으로 AI 튜터(LLM)가 답변 생성
    return None

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[],
    middleware=[education_guardrail],
)

In [ ]:
agent

In [ ]:
agent.invoke({
    "messages": [{"role": "user", "content": "피타고라스의 정리가 이해가 안 돼. 설명해줘."}]
})

In [ ]:
agent.invoke({
    "messages": [{"role": "user", "content": "나 독후감 쓰기 귀찮은데 숙제 대신 써줘."}]
})

In [ ]:
agent.invoke({
    "messages": [{"role": "user", "content": "아 공부하기 싫다. 롤 관련 유튜브 영상이나 찾아줘"}]
})

In [ ]:
agent.invoke({
    "messages": [{"role": "user", "content": "아 너 왜이렇게 바보같냐..."}]
})

###④ After Guardrails

In [ ]:
from langchain.chat_models import init_chat_model

safety_model = init_chat_model("gpt-5-nano")

In [ ]:
from langchain.agents.middleware import after_agent
from langchain.messages import HumanMessage, SystemMessage, AIMessage

@after_agent
def answer_leakage_guardrail(state, runtime):
    """
    AI가 답변을 생성한 '직후', 사용자에게 보여주기 전에 내용을 검사.
    만약 AI가 문제의 정답을 직접적으로 말해버렸다면, 이를 감지하고 수정.
    """

    # 1. 메시지 유효성 검사
    if not state["messages"]: return None
    last_message = state["messages"][-1]

    # 마지막 메시지가 AI의 답변이 아니면 검사할 필요 없음
    if not isinstance(last_message, AIMessage):
        return None

    # 2. 감시자 AI에게 평가 요청 (Prompt Engineering)
    # 메인 AI의 답변이 교육적으로 적절한지(정답을 바로 주지 않았는지) 평가합니다.
    auditor_prompt = f"""
    당신은 엄격한 교육 감독관입니다.
    다음 '튜터의 답변'을 확인하세요.
    답변이 학생을 지도하지 않고 문제의 정답이나 전체 풀이를 직접적으로 제공한다면 'LEAKED'라고 답하세요.
    답변이 적절한 힌트나 설명을 제공한다면 'SAFE'라고 답하세요.

    튜터의 답변: {last_message.content}
    """

    result = safety_model.invoke([{"role": "user", "content": auditor_prompt}])

    # 3단계: 교정 (Correction / Regeneration)
    if "LEAKED" in result.content:
        print(f"🚨 [가드레일 발동] 정답 유출 감지됨! 소크라테스식 교육법으로 답변을 다시 생성합니다.")

        # 원래 사용자의 질문을 가져오기 (문맥 파악용) -> state["messages"][-2]가 보통 사용자 질문
        original_question = state["messages"][-2].content if len(state["messages"]) >= 2 else "사용자 질문 알 수 없음"

        # 교정 모델에게 "정답을 빼고 힌트로 바꿔라"고 지시
        correction_prompt = f"""
        당신은 친절한 AI 튜터입니다.

        절대 정답을 직접 말하지 말고, 학생이 스스로 생각할 수 있도록 유도하는 질문이나 핵심 개념(힌트)만 설명하세요.
        말투는 친절하게 해주세요.

        사용자 질문: {original_question}
        """

        # LLM을 다시 호출하여 새로운 답변 생성 (비용은 1회 더 발생하지만 품질 확보)
        corrected_response = safety_model.invoke(
            [SystemMessage(content="당신은 소크라테스식 교육법을 사용하는 튜터입니다."),
             HumanMessage(content=correction_prompt)]
        )
        print(f"원본: {last_message.content}")
        # 원래의 유출된 답변을 교정된 답변으로 덮어쓰기
        last_message.content = corrected_response.content

    return None


In [ ]:
agent = create_agent(
    model="gpt-5",
    tools=[],
    middleware=[answer_leakage_guardrail],
)

In [ ]:
agent

In [ ]:
agent.invoke({
    "messages": [{"role": "user", "content": "직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 이 문제 너무 어려워. 그냥 정답 알려줘."}]
})

###⑤ Combine multiple Guardrails

before_agent
1. student_safety_middleware : 입력 메세지에서 전화번호/이메일 감지 & 마스킹 처리
2. counseling_escalation_middleware : 심각한 고민/위기 상황 감지 & 상담 이관


In [ ]:
import re

@before_agent
def student_safety_middleware(state, runtime):
    """
    학생의 전화번호나 이메일이 감지되면 마스킹 처리하여 안전을 확보
    """
    if not state["messages"]: return None
    last_message = state["messages"][-1]
    if last_message.type != "human": return None

    content = last_message.content
    original_content = content # 로깅용

    # 전화번호 패턴 (010-XXXX-XXXX 또는 010XXXXXXXX 등)
    phone_pattern = r'01[016789]-?[0-9]{3,4}-?[0-9]{4}'
    # 이메일 패턴
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'

    is_redacted = False

    if re.search(phone_pattern, content):
        content = re.sub(phone_pattern, '<PHONE_REDACTED>', content)
        is_redacted = True

    if re.search(email_pattern, content):
        content = re.sub(email_pattern, '<EMAIL_REDACTED>', content)
        is_redacted = True

    if is_redacted:
        print(f"🔒 [학생 보호] 개인정보가 감지되어 마스킹 처리했습니다.\n원본: {original_content}\n수정: {content}")
        # 내용을 수정하여 LLM에게 전달 (사용자에게 알릴 필요 없이 조용히 처리하거나, 시스템 메시지 추가 가능)
        last_message.content = content

    return None

In [ ]:
ESCALATION_KEYWORDS = ["왕따", "괴롭힘", "우울해", "학교 폭력", "상담 선생님", "사람 불러줘"]

@before_agent(can_jump_to=["end"])
def counseling_escalation_middleware(state, runtime) :
    """
    [Layer 3] 심리적 위기 상황이나 상담 요청이 감지되면 AI 답변을 멈추고 인간 상담사에게 알림을 보냅니다.
    """
    if not state["messages"]: return None
    last_message = state["messages"][-1]

    # 민감한 키워드가 포함되어 있는지 확인
    for keyword in ESCALATION_KEYWORDS:
        if keyword in last_message.content:
            print(f"✋ [상담 이관] 심각한 고민/요청 감지: {keyword}")

            # 여기서 실제로는 상담 교사에게 알림(Slack, Email 등)을 보내는 로직이 들어감
            # send_alert_to_teacher(last_message.content)

            return {
                "messages": [{
                    "role": "assistant",
                    "content": "학생, 많이 힘들었겠구나. 이 문제는 내가 답변하기보다는 전문 상담 선생님이 직접 듣고 도와주시는 게 좋을 것 같아. \n\n지금 바로 상담 선생님께 연결해 드렸으니 잠시만 기다려 줄래? 🍀 (상담실 연결 중...)"
                }],
                "jump_to": "end" # AI 답변 생성 중단
            }
    return None


In [ ]:
# 4중 방어막이 적용된 에이전트
agent = create_agent(
    model="gpt-5-nano",
    tools=[],
    middleware=[
        education_guardrail,             # Layer 1: 입력 필터 (규칙 - 딴짓/부정행위) - before_agent
        student_safety_middleware,       # Layer 2: 개인정보 보호 (전화번호 마스킹) - before_agent
        counseling_escalation_middleware,# Layer 3: 상담 이관 (휴먼 에스컬레이션) - before_agent
        answer_leakage_guardrail         # Layer 4: 출력 필터 (모델 기반 교정) - after_agent
    ],
)

In [ ]:
agent

In [ ]:
agent.invoke({"messages": [{"role": "user", "content": "저 수학 과외 구하고 싶어요. 제 번호 010-1234-5678로 연락 주세요."}]})

In [ ]:
agent.invoke({"messages": [{"role": "user", "content": "나 요즘 학교에서 왕따 당하는 것 같아서 너무 우울해."}]})

#2 . RAG

In [ ]:
!pip install -qU \
    langchain==1.3.9 \
    langchain-core==1.4.7 \
    langchain-openai==1.3.2 \
    langchain-community==0.4.2 \
    langchain-classic==1.0.8 \
    pypdf==6.13.2 \
    chromadb==1.5.9 \
    "langchain-chroma>=0.1.2"

##2-1. 데이터 파일 생성 (manhwa_data.txt)

In [ ]:
# 데이터 파일 생성
manhwa_content = """
[작가: 김동화]
* 작품명: 나의 하늘 (1982년) - 가난하지만 맑은 영혼을 가진 소녀의 첫사랑 이야기.
* 작품명: 요정 핑크 (1987년) - 요정 공주 핑크와 인간 소년 빈의 판타지 로맨스.
* 작품명: 아카시아 (1995년) - 고대 이집트의 전생과 현대의 환생을 오가며 펼쳐지는 무희 아카시아의 격정적인 사랑과 복수를 그린 로맨틱 대서사시.

[작가: 이미라]
* 작품명: 언제나 그대와 (1986년) - 명랑하고 씩씩한 여주인공의 로맨틱 코미디.
* 작품명: 은비가 내리는 나라 (1989년) - 도깨비 공주 은비와 이슬이의 시공간 초월 판타지.
* 작품명: 늘푸른 이야기 (1987년) -고교 시절 우정과 사랑, 성장을 다룬 작품으로, 일상의 소소한 재미와 감동이 있음.

[작가: 황미나]
* 작품명: 굿바이 미스터 블랙 (1983년) - 배신과 복수를 다룬 장대한 서사시.
* 작품명: 불새의 늪 (1984년) - 프랑스 혁명 배경의 시대극 명작.
* 작품명: 레드문 (1990) - 고통받는 자신의 별을 구하기 위해 진정한 구원자로 거듭나는 과정을 그린 한국 SF 만화의 기념비적인 대작

[작가: 신일숙]
* 작품명: 아르미안의 네 딸들 (1986년) - 네 공주의 운명과 왕국을 다룬 대서사 판타지.
* 작품명: 별의 아리아 (1980년) - 음악적 재능을 가진 주인공들의 예술에 대한 열정과 경쟁, 그리고 그 속에 피어나는 사랑을 다룬 감동적인 이야기.
"""

with open("manhwa_data.txt", "w", encoding="utf-8") as f:
    f.write(manhwa_content)

##2-2. RAG 구축 (Pre-Processing)

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# ① Document Load
loader = TextLoader("manhwa_data.txt")
docs = loader.load()

# ② Text Split
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    separators=["\n\n", "\n", " ", ""]
)
splits = text_splitter.split_documents(docs)

print(f"Loaded {len(docs)} documents from {loader.file_path}")
print(f"Split into {len(splits)} chunks of text")

# ③ Embedding  &  ④ DB Store
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory="./chroma_db"
)

In [ ]:
# ⑤ Retriever
retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k": 3}
)

##2-3. RAG 실행 (Runtime)

In [ ]:
template = """당신은 고전 순정만화 전문가입니다. 제공된 문맥을 바탕으로 친절하게 답하세요.
문맥: {context}
질문: {question}
답변:"""
prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(docs):
    print(f"***** 관련 RAG 문서 개수 = {len(docs)}")
    return "\n\n".join(doc.page_content for doc in docs)

# RAG 체인 정의
rag_chain = (
    {"context": RunnablePassthrough() | retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# 대화형 질의
print("질문을 입력하세요 (종료: exit or quit)")
while True:
    question = input("\n질문: ")
    if question.lower() in ['exit', 'quit', '종료']:
        print("종료합니다.")
        break

    response = rag_chain.invoke(question)

    print(f"답변: {response}")
